# Prepare data for single chromosome experiment

We select data progressively

1. Only reads intersecting with DMRs with average coverage in the read region >+ 40 reads.
2. First pair of fails is used for training with train/valid/test split 60/20/20
3. Other four pairs are used for testing


Import libraries and define sources

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
# source = "E:/methyldldata/Curated/melanoma_rrms_2022.07_processed_reads/tumor/COLO829_1.bam"
# data = pd.read_parquet(source, filters=[("chromosome", "==", "chr5"), ("overlapping_with_dmrs", "==", "true")])

In [5]:
source = "X:/methyldldata/Curated/melanoma_rrms_2022.07_processed_reads_with_coverage/"
source_normal = source+"normal/"
source_tumor = source+"tumor/"
normal_parquet_files = [source_normal+x for x in os.listdir(source_normal)]
tumour_parquet_files = [source_tumor+x for x in os.listdir(source_tumor)]

Loading and processing data

In [2]:
def load_and_filter_data(parquet_file_path,target_chromosome, overlapping_with_dmrs="true", minimal_coverage=40):
    data = pd.read_parquet(parquet_file_path, filters=[("chromosome", "==", target_chromosome), ("overlapping_with_dmrs", "==", overlapping_with_dmrs)])
    data["original_file"] = parquet_file_path.split("/")[-1]
    label = 0 if parquet_file_path.split("/")[-2]=="normal" else 1
    data["label"] = label
    data.rename(columns={"seq": "input_ids", "methylation_encoding": "methylation_ids"}, inplace=True)
    return data.loc[data["coverage"]>=minimal_coverage,]

In [3]:
def process_ang_generate_datasets(normal_parquet_files, tumour_parquet_files):
    list_of_chromosomes = ["chr"+str(x) for x in range(1,23)]
    for chr in list_of_chromosomes:
        all_data = [load_and_filter_data(file, chr) for file in set(normal_parquet_files).union(tumour_parquet_files)]
        all_data = pd.concat(all_data)
        first_pair_data = all_data.loc[(all_data["original_file"] == "COLO829BL_1.bam") | (all_data["original_file"] =="COLO829_1.bam"),]
        rest_data = all_data.loc[(all_data["original_file"] != "COLO829BL_1.bam") & (all_data["original_file"] !="COLO829_1.bam"),]
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=1238594)
        train_index, test_index = next(sss.split(first_pair_data, first_pair_data[["label"]]))
        train_data = first_pair_data.iloc[train_index]
        test_data = first_pair_data.iloc[test_index]
        sss_valid = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=1238594)
        train_index, valid_index = next(sss_valid.split(train_data, train_data[["label"]]))
        valid_data = train_data.iloc[valid_index]
        train_data = train_data.iloc[train_index]
        print(chr," Train data shape: ", train_data.shape, f"Fraction of tumor labels: {np.mean(train_data["label"]==1):.2f}")
        print(chr," Valid data shape: ", valid_data.shape, f"Fraction of tumor labels: {np.mean(valid_data["label"]==1):.2f}")
        print(chr," Test data shape: ", test_data.shape, f"Fraction of tumor labels: {np.mean(test_data["label"]==1):.2f}")
        print(chr," Rest data shape: ", rest_data.shape, f"Fraction of tumor labels: {np.mean(rest_data["label"]==1):.2f}")
        save_path = f"../Data/Curated/rrms_dmrs_only_mincov40_corrected/{chr}/"
        if not os.path.exists(save_path):
            os.mkdir(save_path)
        train_data.to_parquet(save_path+"/train.parquet", index = False)
        test_data.to_parquet(save_path+"/test.parquet", index = False)
        valid_data.to_parquet(save_path+"/valid.parquet", index = False)
        rest_data.to_parquet(save_path+"/rest.parquet", index = False)

In [7]:
process_ang_generate_datasets(normal_parquet_files,tumour_parquet_files)

chr1  Train data shape:  (118246, 37) Fraction of tumor labels: 0.56
chr1  Valid data shape:  (39416, 37) Fraction of tumor labels: 0.56
chr1  Test data shape:  (39416, 37) Fraction of tumor labels: 0.56
chr1  Rest data shape:  (618221, 37) Fraction of tumor labels: 0.57
chr2  Train data shape:  (32460, 37) Fraction of tumor labels: 0.59
chr2  Valid data shape:  (10820, 37) Fraction of tumor labels: 0.59
chr2  Test data shape:  (10821, 37) Fraction of tumor labels: 0.59
chr2  Rest data shape:  (120085, 37) Fraction of tumor labels: 0.68
chr3  Train data shape:  (16418, 37) Fraction of tumor labels: 0.76
chr3  Valid data shape:  (5473, 37) Fraction of tumor labels: 0.76
chr3  Test data shape:  (5473, 37) Fraction of tumor labels: 0.76
chr3  Rest data shape:  (66580, 37) Fraction of tumor labels: 0.86
chr4  Train data shape:  (15042, 37) Fraction of tumor labels: 0.58
chr4  Valid data shape:  (5014, 37) Fraction of tumor labels: 0.58
chr4  Test data shape:  (5014, 37) Fraction of tumor l

Preparing data for epigenbert